<a href="https://colab.research.google.com/github/senithimuhandiram/Senithi.20231902.CRM_CW1/blob/main/CRM_CW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q yfinance
import pandas as pd, numpy as np, yfinance as yf, requests, io, os, time

**1.1 Ticker selection and data acquisition**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/CRM_CW"
for d in ["data", "outputs", "figures"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)

Attempting to mount Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directories configured successfully using BASE: /content/drive/MyDrive/CRM_CW


In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
sp = pd.read_html(io.StringIO(html))[0]
sp["Symbol"] = sp["Symbol"].str.replace(".", "-", regex=False)
sp.to_csv(f"{BASE}/data/sp500_constituents.csv", index=False)
tickers = sp["Symbol"].tolist()
print(len(tickers)); sp.head()

503


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [ ]:
parts = []
all_t = tickers + ["^GSPC"]
for i in range(0, len(all_t), 100):
    parts.append(yf.download(all_t[i:i+100], start="2021-01-01", end="2025-01-01",
                             auto_adjust=False, threads=True))
    time.sleep(5)
raw = pd.concat(parts, axis=1)

[*********************100%***********************]  100 of 100 completed
[*********************100%***********************]  100 of 100 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
[*********************100%***********************]  100 of 100 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
[*********************100%***********************]  100 of 100 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 173

In [ ]:
adj   = raw["Adj Close"]
close = raw["Close"]
vol   = raw["Volume"]

adj.to_csv(f"{BASE}/data/adj_close_raw.csv")
close.to_csv(f"{BASE}/data/close_raw.csv")
vol.to_csv(f"{BASE}/data/volume_raw.csv")
sp.to_csv(f"{BASE}/data/sp500_constituents.csv", index=False)

**1.2 Data validation and alignment**

In [ ]:
stocks = adj.drop(columns="^GSPC")
missing = stocks.isna().mean()
print("Failed completely:", (missing == 1).sum())
print("Partly missing:", ((missing > 0) & (missing < 1)).sum())
missing.sort_values(ascending=False).head(20)

Failed completely: 5
Partly missing: 10


,0
Ticker,
HONA,1.000000
VYLR,1.000000
FDXF,1.000000
SNDK,1.000000
Q,1.000000
GEV,0.807960
SOLV,0.806965
RDDT,0.803980
VLTO,0.688557


In [ ]:
failed = missing[missing == 1].index.tolist()
short  = missing[(missing > 0.10) & (missing < 1)].index.tolist()
keep   = missing[missing <= 0.10].index.tolist() + ["^GSPC"]

adj_c   = adj[keep].ffill(limit=2)
close_c = close[keep].ffill(limit=2)
vol_c   = vol[keep].fillna(0)

print("Failed:", len(failed), failed)
print("Dropped (short history):", len(short), short)
print("Final stocks:", len(keep) - 1)
print("Missing left:", adj_c.isna().sum().sum())

Failed: 5 ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']
Dropped (short history): 7 ['CEG', 'GEHC', 'GEV', 'HOOD', 'RDDT', 'SOLV', 'VLTO']
Final stocks: 491
Missing left: 165


In [ ]:
adj_c.to_csv(f"{BASE}/data/adj_close_clean.csv")
close_c.to_csv(f"{BASE}/data/close_clean.csv")
vol_c.to_csv(f"{BASE}/data/volume_clean.csv")

summary = pd.DataFrame({"Step": ["In Wikipedia list", "Failed download", "Dropped: >10% missing", "Final sample"],
                        "Stocks": [len(tickers), len(failed), len(short), len(keep) - 1]})
summary.to_csv(f"{BASE}/outputs/data_cleaning_summary.csv", index=False)
summary

,Step,Stocks
0,In Wikipedia list,503
1,Failed download,5
2,Dropped: >10% missing,7
3,Final sample,491
